In [1]:
import findspark
findspark.init("C:\Apps\spark-3.0.3-bin-hadoop2.7")
import pyspark
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.regression import DecisionTreeRegressor
import mlflow
from urllib.parse import urlparse
import json
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit
from hyperopt import Trials

from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import RandomForestRegressor

In [2]:
# def create_train_test():
spark = SparkSession.builder \
               .appName('taxi_fare_amount_prediction') \
                .master('spark://vps00:7077')  \
                .config('spark.sql.execution.arrow.pyspark.enabled', True) \
                .config('spark.sql.session.timeZone', 'UTC') \
                .config('spark.driver.memory','6G') \
                .config('spark.ui.showConsoleProgress', True) \
                .config('spark.sql.repl.eagerEval.enabled', True) \
                .getOrCreate()
#read csv_file in pyspark
# trip_fare_amount_data = spark.read.options(header=True).csv('../../data/processed/data.csv',inferSchema = True)
trip_fare_amount_data=spark.read.parquet("../../data/processed/data.parquet")

In [5]:
# print(trip_fare_amount_data.printSchema())
input_column_name_lambda_fun = lambda col_name: col_name if(col_name !="fare_amount") else None
input_column_names = input_column_name_lambda_fun(trip_fare_amount_data.columns)


In [6]:

vectorAssembler = VectorAssembler(inputCols = input_column_names, outputCol = "features")
vpp_sdf = vectorAssembler.transform(trip_fare_amount_data)
# vpp_sdf.show(2, False)
vpp_sdf_final = vpp_sdf.select(["features","fare_amount"])
# vpp_sdf_final = vpp_sdf_final.withColumnRenamed("fare_amount","label")
splits = vpp_sdf_final.randomSplit([0.7,0.3])
train_df = splits[0]
test_df = splits[1]

In [5]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

Decision_tree_reg = DecisionTreeRegressor(featuresCol = "features",
                                          labelCol = "fare_amount",
                                          maxMemoryInMB = 5000)

grid = ParamGridBuilder()\
    .addGrid(Decision_tree_reg.minInstancesPerNode, [1])\
    .addGrid(Decision_tree_reg.maxBins, [32]).build()

In [9]:

r2_evaluator = RegressionEvaluator(labelCol="fare_amount", \
    predictionCol="prediction", metricName="r2")
cv = CrossValidator(estimator=Decision_tree_reg, estimatorParamMaps=grid, \
    evaluator=r2_evaluator,
    parallelism=3)

In [10]:
Decision_tree_reg_model = cv.fit(train_df)
# 79m 56.2s

In [15]:
Decision_tree_reg_model.avgMetrics[0] * 100

0.4161690083998841

In [16]:

randomforest_reg = RandomForestRegressor(featuresCol = "features",
                                          labelCol = "fare_amount",
                                          maxMemoryInMB = 5000)

grid = ParamGridBuilder()\
    .addGrid(randomforest_reg.minInstancesPerNode, [1])\
    .addGrid(randomforest_reg.maxBins, [32]).build()

r2_evaluator = RegressionEvaluator(labelCol="fare_amount", \
    predictionCol="prediction", metricName="r2")
cv = CrossValidator(estimator=randomforest_reg, estimatorParamMaps=grid, \
    evaluator=r2_evaluator,
    parallelism=2)


In [17]:
%%time
randomforest_reg = cv.fit(train_df)


In [19]:
randomforest_reg.avgMetrics[0] * 100

0.8308194642174879

In [7]:
def train_tree(maxBins, minInstancesPerNode):
    randomforest_reg = RandomForestRegressor(featuresCol = "features",
                                          labelCol = "fare_amount",
                                          maxBins = maxBins , minInstancesPerNode=minInstancesPerNode,
                                          maxMemoryInMB = 5000)
    randomforest_reg_model = randomforest_reg.fit(train_df)
    r2_evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="r2")
    predictions = randomforest_reg_model.transform(test_df)
    validation_metric = r2_evaluator.evaluate(predictions)
    
    return randomforest_reg_model, validation_metric

def train_with_hyperopt(params):
    print(params)
    maxBins =int(params['maxBins']) 
    minInstancesPerNode=int(params['minInstancesPerNode']) 
    model, r2_score = train_tree(maxBins, minInstancesPerNode)
    
    loss =  - r2_score
    return {'loss': loss, 'status': STATUS_OK}


In [8]:
def evaluate_model(model,val_r2_score):
    pred_results =  model.transform(test_df)
    rmse_evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="rmse")
    rmse_eval = rmse_evaluator.evaluate(pred_results)

    mse_evaluator = RegressionEvaluator(labelCol="fare_amount", predictionCol="prediction", metricName="mse")
    mse_eval = mse_evaluator.evaluate(pred_results)

    mae_evaluator = RegressionEvaluator(
                labelCol="fare_amount", predictionCol="prediction", metricName="mae")
    mae_eval = mae_evaluator.evaluate(pred_results)

    decision_Tree_reg = {}
    decision_Tree_reg['rmse'] = rmse_eval
    decision_Tree_reg['mae'] = mae_eval
    decision_Tree_reg['r2'] = val_r2_score
    decision_Tree_reg['mse'] = mse_eval

    return decision_Tree_reg

In [7]:
space = {
    'maxBins': hp.uniform('maxBins', 60, 66),
    'minInstancesPerNode':hp.uniform('minInstancesPerNode', 50, 56),
}


# algo=tpe.suggest

# trails =Trials()

In [8]:
best_params = fmin(
            fn=train_with_hyperopt,
            space=space,
            algo=tpe.suggest,
            max_evals=3,
            trials=Trials()
            )
best_params

{'maxBins': 65.74860083239379, 'minInstancesPerNode': 50.644123192046685}                                              
{'maxBins': 65.18439835360766, 'minInstancesPerNode': 51.132936317395696}                                              
{'maxBins': 65.60410981354126, 'minInstancesPerNode': 55.95814388927974}                                               
100%|████████████████████████████████████████████| 3/3 [1:21:21<00:00, 1627.27s/trial, best loss: -0.09553574023308387]


{'maxBins': 65.74860083239379, 'minInstancesPerNode': 50.644123192046685}

In [ ]:
-0.09553574023308387

In [ ]:
{'maxBins': 275.0, 'maxDepth': 7.0, 'minInstancesPerNode': 255.0, 'numTrees': 18.0}                                    
{'maxBins': 270.0, 'maxDepth': 8.0, 'minInstancesPerNode': 184.0, 'numTrees': 19.0}                                    
2/2 [45:00<00:00, 1350.30s/trial, best loss: -0.01716529127172417]
{'maxBins': 275.0,
 'maxDepth': 7.0,
 'minInstancesPerNode': 255.0,
 'numTrees': 18.0}

In [ ]:
{'maxBins': 174.0, 'maxDepth': 6.0, 'minInstancesPerNode': 90.0, 'numTrees': 14.0}                                     
{'maxBins': 179.0, 'maxDepth': 6.0, 'minInstancesPerNode': 130.0, 'numTrees': 24.0}                                    
2/2 [50:58<00:00, 1529.16s/trial, best loss: -0.12175887279952868]
{'maxBins': 179.0,
 'maxDepth': 6.0,
 'minInstancesPerNode': 130.0,
 'numTrees': 24.0}

In [ ]:
{'maxBins': 62.0, 'minInstancesPerNode': 55.0}                                                         
{'maxBins': 65.0, 'minInstancesPerNode': 58.0}                                                         
2/2 [46:59<00:00, 1409.52s/trial, best loss: -0.8896598963749668]
{'maxBins': 65.0, 'minInstancesPerNode': 58.0}

In [ ]:
{'maxBins': 63.0, 'minInstancesPerNode': 81.0}                                                                                  
{'maxBins': 61.0, 'minInstancesPerNode': 86.0}                                                                      
2/2 [53:37<00:00, 1608.72s/trial, best loss: -0.004807170672447958]
{'maxBins': 63.0, 'minInstancesPerNode': 81.0}

In [ ]:
{'maxBins': 62.0, 'minInstancesPerNode': 56.0}                                                    
{'maxBins': 62.0, 'minInstancesPerNode': 184.0}                                                                           
2/2 [50:33<00:00, 1516.88s/trial, best loss: -0.9210733526931156]
{'maxBins': 62.0, 'minInstancesPerNode': 56.0}

In [9]:
%%time
matrics = {}
initial_model, val_metric = train_tree(maxBins=32, minInstancesPerNode=1)

matrics['model_no_hyperparamter'] = evaluate_model(initial_model,val_metric)

print("Intial model score on test dataset")
print(matrics['model_no_hyperparamter'])

Intial model score on test dataset
{'rmse': 34.633839582280025, 'mae': 1.5450094647765817, 'r2': 0.09287411015799962, 'mse': 1199.5028442111068}
Wall time: 30min 5s


In [10]:
randomforest_reg = RandomForestRegressor(featuresCol = "features",
                                          labelCol = "fare_amount",
                                          maxMemoryInMB = 5000)

grid = ParamGridBuilder()\
    .addGrid(randomforest_reg.minInstancesPerNode, [1])\
    .addGrid(randomforest_reg.maxBins, [32]).build()

rmse_evaluator = RegressionEvaluator(labelCol="fare_amount", \
    predictionCol="prediction", metricName="r2")
cv = CrossValidator(estimator=randomforest_reg, estimatorParamMaps=grid, \
    evaluator=rmse_evaluator,
    parallelism=2)


In [11]:
%%time
randomforest_reg_model = cv.fit(train_df)

Wall time: 1h 4min 6s


In [13]:
randomforest_reg_model.avgMetrics[0] * 100

27.044352533567473

In [10]:
# matrics = {}
# matrics['model_no_hyperparamter'] = {'rmse': 152.26312061801195, 'mae': 1.4890066590120221, 'r2': 0.005513526489589582, 'mse': 23184.057900335247}

In [ ]:
# print('trials:')
# for trial in trails.trials:
#     print (trial)

In [14]:
best_minInstancesPerNode = int(62.0)
best_maxBins = int(56.0)

In [15]:
final_model, val_rmse_score = train_tree(best_minInstancesPerNode, best_maxBins)
matrics['model_hyperparamter'] = evaluate_model(final_model,val_rmse_score)

In [16]:
matrics

{'model_no_hyperparamter': {'rmse': 34.633839582280025,
  'mae': 1.5450094647765817,
  'r2': 0.09287411015799962,
  'mse': 1199.5028442111068},
 'model_hyperparamter': {'rmse': 34.58064735516762,
  'mae': 1.6087358986558713,
  'r2': 0.09565838086486356,
  'mse': 1195.8211715024613}}

In [14]:
matrics

{'model_no_hyperparamter': {'rmse': 152.26312061801195,
  'mae': 1.4890066590120221,
  'r2': 0.005513526489589582,
  'mse': 23184.057900335247},
 'model_hyperparamter': {'rmse': 152.2442078596439,
  'mae': 1.3960079598023354,
  'r2': 0.005760563515850969,
  'mse': 23178.298826810456}}

In [17]:
randomforest_reg = RandomForestRegressor(featuresCol = "features",
                                          labelCol = "fare_amount",
                                          maxMemoryInMB = 5000)

grid = ParamGridBuilder()\
    .addGrid(randomforest_reg.minInstancesPerNode, [best_minInstancesPerNode])\
    .addGrid(randomforest_reg.maxBins, [best_maxBins]).build()

rmse_evaluator = RegressionEvaluator(labelCol="fare_amount", \
    predictionCol="prediction", metricName="r2")
cv = CrossValidator(estimator=randomforest_reg, estimatorParamMaps=grid, \
    evaluator=rmse_evaluator,
    parallelism=2)


In [18]:
%time randomforest_reg_model = cv.fit(train_df)

Wall time: 1h 15min 24s


In [19]:
randomforest_reg_model.avgMetrics[0] * 100

29.823512198279396

In [62]:
model_path = "C:/Data/taxi_trips/final_project_folder/taxi_trips/models/randomForest/"
final_model.write().overwrite().save(model_path)

In [17]:
randomforest_reg_model.avgMetrics[0] * 100

33.82277762848432

In [21]:
Decision_tree_reg_model.avgMetrics[0] * 100

30.147746242639933

In [27]:
trip_fare_amount_data.printSchema()

root
 |-- trip_distance: double (nullable = true)
 |-- pulocationid: double (nullable = true)
 |-- dolocationid: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- passenger_count_imputed: double (nullable = true)
 |-- tpep_pickup_hour: integer (nullable = true)
 |-- tpep_pickup_minute: integer (nullable = true)
 |-- tpep_pickup_year: integer (nullable = true)
 |-- tpep_pickup_month: integer (nullable = true)
 |-- tpep_pickup_date: integer (nullable = true)
 |-- tpep_dropoff_hour: integer (nullable = true)
 |-- tpep_dropoff_minute: integer (nullable = true)
 |-- tpep_dropoff_year: integer (nullable = true)
 |-- tpep_dropoff_month: integer (nullable = true)
 |-- tpep_dropoff_date: integer (nullable = true)



In [93]:
trip_fare_amount_data.columns

['trip_distance',
 'pulocationid',
 'dolocationid',
 'fare_amount',
 'passenger_count_imputed',
 'tpep_pickup_hour',
 'tpep_pickup_minute',
 'tpep_pickup_year',
 'tpep_pickup_month',
 'tpep_pickup_date',
 'tpep_dropoff_hour',
 'tpep_dropoff_minute',
 'tpep_dropoff_year',
 'tpep_dropoff_month',
 'tpep_dropoff_date']

In [95]:
data_dict = [{
'trip_distance':20,
 'pulocationid':1,
 'dolocationid':2,
 'passenger_count_imputed':2,
 'tpep_pickup_hour':14,
 'tpep_pickup_minute':17,
 'tpep_pickup_year':2020,
 'tpep_pickup_month':8,
 'tpep_pickup_date':1,
 'tpep_dropoff_hour':15,
 'tpep_dropoff_minute':15,
 'tpep_dropoff_year':2020,
 'tpep_dropoff_month':8,
 'tpep_dropoff_date':1
}]


In [96]:
data_dict

[{'trip_distance': 20,
  'pulocationid': 1,
  'dolocationid': 2,
  'passenger_count_imputed': 2,
  'tpep_pickup_hour': 14,
  'tpep_pickup_minute': 17,
  'tpep_pickup_year': 2020,
  'tpep_pickup_month': 8,
  'tpep_pickup_date': 1,
  'tpep_dropoff_hour': 15,
  'tpep_dropoff_minute': 15,
  'tpep_dropoff_year': 2020,
  'tpep_dropoff_month': 8,
  'tpep_dropoff_date': 1}]

In [97]:
spark_dataframe = spark.createDataFrame(data=data_dict)

In [98]:
from pyspark.ml.regression import RandomForestRegressionModel
model_path = "../../models/randomForest/"
model = RandomForestRegressionModel.load(model_path)
    

In [99]:
vectorAssembler = VectorAssembler(inputCols = spark_dataframe.columns, outputCol = "features")

In [100]:
vpp_sdf = vectorAssembler.transform(spark_dataframe)

In [101]:
selected_feauture = vpp_sdf.select("features")

In [102]:
vpp_sdf

dolocationid,passenger_count_imputed,pulocationid,tpep_dropoff_date,tpep_dropoff_hour,tpep_dropoff_minute,tpep_dropoff_month,tpep_dropoff_year,tpep_pickup_date,tpep_pickup_hour,tpep_pickup_minute,tpep_pickup_month,tpep_pickup_year,trip_distance,features
2,2,1,1,15,15,8,2020,1,14,17,8,2020,20,"[2.0,2.0,1.0,1.0,..."


In [106]:
prediction = model.transform(selected_feauture)

In [108]:
prediction.select("prediction").toJSON().map(lambda j: json.loads(j)).collect()

[{'prediction': 13.380983856172898}]

In [111]:
from geopy import distance

In [110]:
!pip install geopy

In [113]:
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="test")

In [150]:
location_1 = geolocator.geocode("Arrochar")

In [151]:
print((location_1.latitude, location_1.longitude))

(56.1954653, -4.7480746)


In [152]:
location_2 = geolocator.geocode("Fort Wadsworth")

In [153]:
print((location_2.latitude, location_2.longitude))

(40.60076325, -74.05763900439601)


In [154]:
d = distance.distance((location_1.latitude, location_1.longitude),(location_2.latitude, location_2.longitude))
float(d.miles)

3212.2111320252593

In [144]:
d = distance.distance((location_1.latitude, location_1.longitude),(location_2.latitude, location_2.longitude))
d.miles

192.20770319923625